# 📊 Módulo 3: Gestión, Administración y Optimización Analítica de Deudas
**Responsable OED:** Catalina Moreno — *Ingeniera Industrial (Líder de Proyectos y Analítica)*  
**Marco Normativo Colombiano:** Circular Básica Jurídica SFC (Tasas $EA \leftrightarrow EM$), Art. 884 C. de Co. (Tasa de Usura SFC), Ley 1328 de 2009 (Derecho a prepago sin sanción ni penalidad).

---

### 🎯 Objetivo del Módulo (Proyecto "Cero Deudas")
Aplicar **Investigación de Operaciones, Gestión de Proyectos y Analítica Prescriptiva** para diagnosticar el pasivo de un hogar colombiano y diseñar la ruta crítica óptima de liberación de flujo de caja mediante:
1. **Diagnóstico de Endeudamiento y Control Legal de Usura SFC:** Semáforo de Índice de Endeudamiento ($IE$) y verificación de que ninguna tasa supere $1.5 \times \text{IBC}$.
2. **Motor Comparativo de Desendeudamiento con *Roll-Over*:**
   * **Método Avalancha (Óptimo Matemático):** Prioriza abonos a la obligación de mayor tasa Efectiva Anual ($EA$), minimizando el pago total de intereses.
   * **Método Bola de Nieve (Óptimo Conductual):** Prioriza abonos a la deuda de menor saldo insoluto, logrando victorias tempranas.
3. **Simulador Ley 1328 de 2009 (Abonos Extraordinarios a Capital) y Compra de Cartera:** Comparación cuantitativa entre **Reducción de Plazo** vs. **Reducción de Cuota** al abonar primas o cesantías.
4. **Diagrama de Gantt del Proyecto Cero Deudas:** Cronograma visual de extinción de cada obligación.

In [ ]:
import sys
from pathlib import Path

# Asegurar que el directorio raíz del proyecto esté en sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.finance import cargar_configuracion, ea_a_em
from src.mod1_presupuesto import diagnosticar_presupuesto_y_flujo_caja
from src.mod3_gestion_deudas import (
    optimizar_portafolio_deudas,
    simular_abono_extraordinario_ley_1328,
    simular_compra_cartera,
)

cfg = cargar_configuracion()
print("✅ Configuración regulatoria OED cargada correctamente.")
print(f"📌 Tasa de Usura SFC vigente (Consumo): {cfg['parametros_macro_colombia']['tasa_usura_ea_consumo']*100:.2f}% EA")

## 1. Inventario de Pasivos y Diagnóstico de Endeudamiento (Conexión con Módulo 1)
Tomamos el perfil semilla del taller (salario bruto de **$\$6.500.000\text{ COP}$**) liquidado por el Módulo 1 y registramos un portafolio realista de 4 obligaciones financieras en Colombia:

In [ ]:
# Portafolio de deudas del caso de estudio en Colombia
portafolio_deudas = [
    {
        "nombre": "1. Tarjeta de Crédito Gold (Compras y Avances)",
        "saldo_cop": 4_800_000.0,
        "tasa_ea": 0.2580,  # 25.80% EA (Cercana a usura)
        "cuota_mensual_cop": 320_000.0,
        "seguro_cuota_manejo_cop": 28_000.0,
    },
    {
        "nombre": "2. Crédito Rotativo / tienda departamental",
        "saldo_cop": 1_900_000.0,
        "tasa_ea": 0.2250,  # 22.50% EA
        "cuota_mensual_cop": 160_000.0,
        "seguro_cuota_manejo_cop": 14_500.0,
    },
    {
        "nombre": "3. Crédito de Libre Inversión Bancario",
        "saldo_cop": 9_500_000.0,
        "tasa_ea": 0.1680,  # 16.80% EA
        "cuota_mensual_cop": 380_000.0,
        "seguro_cuota_manejo_cop": 18_000.0,
    },
    {
        "nombre": "4. Crédito Educativo / Libranza",
        "saldo_cop": 6_200_000.0,
        "tasa_ea": 0.1240,  # 12.40% EA
        "cuota_mensual_cop": 240_000.0,
        "seguro_cuota_manejo_cop": 9_500.0,
    },
]

cuotas_totales_mes = sum(d["cuota_mensual_cop"] + d["seguro_cuota_manejo_cop"] for d in portafolio_deudas)

# Conexión con Módulo 1 (Nidia) para obtener el Ingreso Neto y el Excedente Mensual
diag_presupuesto = diagnosticar_presupuesto_y_flujo_caja(
    ingreso_bruto_mensual_cop=6_500_000.0,
    gastos_fijos_vitales_cop=2_500_000.0,
    gastos_variables_estilo_vida_cop=850_000.0,
    gastos_hormiga_mensual_cop=250_000.0,
    cuotas_deuda_mensual_cop=cuotas_totales_mes,
    liquidez_actual_fondo_emergencia_cop=6_000_000.0,
)

ingreso_neto = diag_presupuesto["ingreso_neto_mensual_cop"]
abono_extra_disponible = max(350_000.0, round(diag_presupuesto["excedente_mensual_cop"] * 0.60, -3))

res_deudas = optimizar_portafolio_deudas(
    ingreso_neto_mensual_cop=ingreso_neto,
    abono_extra_mensual_cop=abono_extra_disponible,
    lista_deudas=portafolio_deudas,
)

print(f"💰 Ingreso Neto Mensual (Módulo 1): ${ingreso_neto:,.0f} COP")
print(f"💳 Saldo Total de Deudas:           ${res_deudas['saldo_total_deudas_cop']:,.0f} COP")
print(f"📆 Cuota Mensual Total Actual:      ${res_deudas['cuota_mensual_total_actual_cop']:,.0f} COP")
print(f"🚦 Índice de Endeudamiento:         {res_deudas['indice_endeudamiento_pct']:.2f}% ({res_deudas['semaforo_endeudamiento']})")
print(f"🚀 Abono Extra Mensual Asignado:    ${abono_extra_disponible:,.0f} COP")

df_inventario = pd.DataFrame(portafolio_deudas)
df_inventario["tasa_em_pct"] = df_inventario["tasa_ea"].apply(lambda x: round(ea_a_em(x) * 100, 2))
df_inventario["tasa_ea_pct"] = (df_inventario["tasa_ea"] * 100).round(2)
df_inventario["cumple_usura_sfc"] = [v["mensaje_legal"] for v in res_deudas["validaciones_usura_sfc"]]
df_inventario[["nombre", "saldo_cop", "tasa_ea_pct", "tasa_em_pct", "cuota_mensual_cop", "seguro_cuota_manejo_cop", "cumple_usura_sfc"]]

## 2. Investigación de Operaciones: Método Avalancha vs. Bola de Nieve con *Roll-Over*
Comparamos las tres trayectorias bajo el mismo presupuesto mensual total:
* **Plan Base (Pago Mínimo):** Pagar únicamente las cuotas mínimas pactadas sin abonos extra.
* **Método Avalancha (Óptimo Matemático):** Ataca primero la deuda con mayor tasa $EA$ (*Tarjeta de Crédito al 25.8% EA*) y aplica *roll-over* al extinguir cada obligación.
* **Método Bola de Nieve (Óptimo Conductual):** Ataca primero la deuda más pequeña (*Crédito Rotativo de $\$1.900.000$*) para liberar victorias psicológicas rápidas.

In [ ]:
comp = res_deudas["comparativa_estrategias"]
df_comparativa = pd.DataFrame([
    {
        "Estrategia": "1. Pago Mínimo (Sin Abonos Extra)",
        "Meses hasta Cero Deuda": comp["pago_minimo_actual"]["meses_totales_para_cero_deuda"],
        "Intereses Totales (COP)": comp["pago_minimo_actual"]["total_intereses_pagados_cop"],
        "Seguros y Cuotas Manejo (COP)": comp["pago_minimo_actual"]["total_seguros_y_manejo_pagados_cop"],
        "Costo Financiero Total (COP)": comp["pago_minimo_actual"]["costo_financiero_total_cop"],
    },
    {
        "Estrategia": "2. Bola de Nieve (Menor Saldo Primero)",
        "Meses hasta Cero Deuda": comp["metodo_bola_de_nieve_conductual"]["meses_totales_para_cero_deuda"],
        "Intereses Totales (COP)": comp["metodo_bola_de_nieve_conductual"]["total_intereses_pagados_cop"],
        "Seguros y Cuotas Manejo (COP)": comp["metodo_bola_de_nieve_conductual"]["total_seguros_y_manejo_pagados_cop"],
        "Costo Financiero Total (COP)": comp["metodo_bola_de_nieve_conductual"]["costo_financiero_total_cop"],
    },
    {
        "Estrategia": "3. Avalancha (Mayor Tasa EA Primero - Óptimo)",
        "Meses hasta Cero Deuda": comp["metodo_avalancha_optimo_matematico"]["meses_totales_para_cero_deuda"],
        "Intereses Totales (COP)": comp["metodo_avalancha_optimo_matematico"]["total_intereses_pagados_cop"],
        "Seguros y Cuotas Manejo (COP)": comp["metodo_avalancha_optimo_matematico"]["total_seguros_y_manejo_pagados_cop"],
        "Costo Financiero Total (COP)": comp["metodo_avalancha_optimo_matematico"]["costo_financiero_total_cop"],
    },
])

df_comparativa["Ahorro vs. Pago Mínimo (COP)"] = (
    comp["pago_minimo_actual"]["costo_financiero_total_cop"] - df_comparativa["Costo Financiero Total (COP)"]
)
display(df_comparativa)

# Construir curvas de evolución de saldo total mes a mes
df_hist_base = pd.DataFrame(comp["pago_minimo_actual"]["historial_mensual"])
df_hist_base["Estrategia"] = "Pago Mínimo Actual"

df_hist_nieve = pd.DataFrame(comp["metodo_bola_de_nieve_conductual"]["historial_mensual"])
df_hist_nieve["Estrategia"] = "Bola de Nieve (Conductual)"

df_hist_aval = pd.DataFrame(comp["metodo_avalancha_optimo_matematico"]["historial_mensual"])
df_hist_aval["Estrategia"] = "Avalancha (Óptimo Matemático)"

df_trayectorias = pd.concat([df_hist_base, df_hist_nieve, df_hist_aval], ignore_index=True)

fig_curvas = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Curva de Extinción del Saldo Total (COP)",
        "Costo Financiero Acumulado: Intereses + Seguros (COP)",
    ),
)

colores = {
    "Pago Mínimo Actual": "#EF553B",
    "Bola de Nieve (Conductual)": "#FFA15A",
    "Avalancha (Óptimo Matemático)": "#00CC96",
}

for est, grupo in df_trayectorias.groupby("Estrategia"):
    fig_curvas.add_trace(
        go.Scatter(
            x=grupo["mes"],
            y=grupo["saldo_total_cop"],
            mode="lines",
            name=est,
            line=dict(color=colores[est], width=3),
        ),
        row=1,
        col=1,
    )
    fig_curvas.add_trace(
        go.Scatter(
            x=grupo["mes"],
            y=grupo["intereses_acumulados_cop"] + grupo["seguros_acumulados_cop"],
            mode="lines",
            name=est,
            showlegend=False,
            line=dict(color=colores[est], width=3, dash="dot"),
        ),
        row=1,
        col=2,
    )

fig_curvas.update_layout(
    title_text=f"Proyecto Cero Deudas (Catalina): Ahorro Total con Avalancha = ${res_deudas['ahorro_intereses_cop']:,.0f} COP ({res_deudas['meses_ahorrados_con_avalancha']} meses menos)",
    template="plotly_white",
    height=460,
    legend=dict(orientation="h", y=-0.2),
)
fig_curvas.update_xaxes(title_text="Mes del Proyecto")
fig_curvas.update_yaxes(title_text="Pesos Colombianos (COP)")
fig_curvas.show()

## 3. Diagrama de Gantt del "Proyecto Cero Deudas" (Ruta Crítica de Liberación de Caja)
Como ingeniera industrial y líder de proyectos, visualizamos el cronograma de extinción de cada obligación comparando cuándo termina cada crédito bajo el **Método Avalancha** frente al **Método Bola de Nieve**.

In [ ]:
cron_aval = comp["metodo_avalancha_optimo_matematico"]["cronograma_extincion"]
cron_nieve = comp["metodo_bola_de_nieve_conductual"]["cronograma_extincion"]
cron_base = comp["pago_minimo_actual"]["cronograma_extincion"]

registros_gantt = []
for item in cron_base:
    registros_gantt.append({"Obligación": item["obligacion"], "Estrategia": "Pago Mínimo", "Mes Extinción": item["mes_extincion"]})
for item in cron_nieve:
    registros_gantt.append({"Obligación": item["obligacion"], "Estrategia": "Bola de Nieve", "Mes Extinción": item["mes_extincion"]})
for item in cron_aval:
    registros_gantt.append({"Obligación": item["obligacion"], "Estrategia": "Avalancha (Óptimo)", "Mes Extinción": item["mes_extincion"]})

df_gantt = pd.DataFrame(registros_gantt)

fig_gantt = px.bar(
    df_gantt,
    x="Mes Extinción",
    y="Obligación",
    color="Estrategia",
    barmode="group",
    orientation="h",
    text="Mes Extinción",
    color_discrete_map={
        "Pago Mínimo": "#EF553B",
        "Bola de Nieve": "#FFA15A",
        "Avalancha (Óptimo)": "#00CC96",
    },
    title="Diagrama de Gantt del Proyecto Cero Deudas: Mes Exacto de Extinción por Obligación",
    template="plotly_white",
    height=450,
)
fig_gantt.update_traces(texttemplate="Mes %{text}", textposition="outside")
fig_gantt.update_xaxes(title_text="Meses desde Hoy hasta Extinción Total")
fig_gantt.show()

## 4. Simulador Ley 1328 de 2009: Abono Extraordinario a Capital (Reducción de Plazo vs. Cuota) y Compra de Cartera
En Colombia, la **Ley 1328 de 2009** garantiza el derecho a realizar prepagos o abonos extraordinarios a capital en cualquier momento sin sanción ni penalidad, permitiendo elegir entre:
1. **Reducción de Plazo** (manteniendo la cuota fija actual).
2. **Reducción de Cuota** (manteniendo el plazo restante original).

Simulamos el efecto de destinar una **Prima de Servicios de $\$2.500.000\text{ COP}$** como abono directo a capital en el Crédito de Libre Inversión ($\$9.500.000\text{ COP}$ al $16.8\%\text{ EA}$ con 36 meses restantes), y evaluamos también una **Compra de Cartera** consolidada al $13.5\%\text{ EA}$:

In [ ]:
# 1. Simulación Ley 1328 de 2009 sobre el Crédito de Libre Inversión
sim_prepago = simular_abono_extraordinario_ley_1328(
    saldo_actual_cop=9_500_000.0,
    tasa_ea=0.1680,
    plazo_restante_meses=36,
    abono_extra_capital_cop=2_500_000.0,
)

print("=== SIMULACIÓN ABONO EXTRAORDINARIO A CAPITAL (LEY 1328 DE 2009) ===")
print(f"Cuota Mensual Original: ${sim_prepago['cuota_mensual_original_cop']:,.0f} COP (Plazo: 36 meses)")
print(
    f"👉 Opción A (Reducción de Plazo): Nuevo plazo = {sim_prepago['opcion_reduccion_plazo']['nuevo_plazo_meses']} meses "
    f"({sim_prepago['opcion_reduccion_plazo']['meses_ahorrados']} meses ahorrados) | "
    f"Ahorro en intereses = ${sim_prepago['opcion_reduccion_plazo']['ahorro_intereses_cop']:,.0f} COP"
)
print(
    f"👉 Opción B (Reducción de Cuota): Nueva cuota = ${sim_prepago['opcion_reduccion_cuota']['nueva_cuota_mensual_cop']:,.0f} COP "
    f"(Libera ${sim_prepago['opcion_reduccion_cuota']['flujo_mensual_liberado_cop']:,.0f} COP/mes) | "
    f"Ahorro en intereses = ${sim_prepago['opcion_reduccion_cuota']['ahorro_intereses_cop']:,.0f} COP"
)

# 2. Evaluación de Compra de Cartera Unificada
sim_cartera = simular_compra_cartera(
    lista_deudas=portafolio_deudas,
    tasa_ea_compra_cartera=0.1350,  # Oferta de compra de cartera al 13.5% EA
    plazo_meses_nuevo=24,
)

print("\n=== EVALUACIÓN DE COMPRA DE CARTERA CONSOLIDADA (13.5% EA a 24 meses) ===")
print(f"Saldo Unificado:              ${sim_cartera['saldo_unificado_cop']:,.0f} COP")
print(f"Cuota Mensual Anterior Total: ${sim_cartera['cuota_mensual_anterior_cop']:,.0f} COP")
print(f"Nueva Cuota Unificada Total:  ${sim_cartera['nueva_cuota_mensual_total_cop']:,.0f} COP")
print(f"Flujo Mensual Liberado:       ${sim_cartera['flujo_caja_mensual_liberado_cop']:,.0f} COP/mes")
print(f"Ahorro Neto Costo Financiero: ${sim_cartera['ahorro_neto_costo_financiero_cop']:,.0f} COP (Viable: {sim_cartera['es_viable_financieramente']})")

fig_ley1328 = px.bar(
    x=["Opción A: Reducción de Plazo (Óptimo Financiero)", "Opción B: Reducción de Cuota (Alivio de Caja)"],
    y=[
        sim_prepago["opcion_reduccion_plazo"]["ahorro_intereses_cop"],
        sim_prepago["opcion_reduccion_cuota"]["ahorro_intereses_cop"],
    ],
    text=[
        f"${sim_prepago['opcion_reduccion_plazo']['ahorro_intereses_cop']:,.0f} COP ({sim_prepago['opcion_reduccion_plazo']['meses_ahorrados']} meses menos)",
        f"${sim_prepago['opcion_reduccion_cuota']['ahorro_intereses_cop']:,.0f} COP (Libera ${sim_prepago['opcion_reduccion_cuota']['flujo_mensual_liberado_cop']:,.0f}/mes)",
    ],
    color=["Reducción de Plazo", "Reducción de Cuota"],
    color_discrete_sequence=["#00CC96", "#636EFA"],
    labels={"x": "Alternativa Ley 1328 de 2009", "y": "Ahorro Total en Intereses (COP)"},
    title="Impacto de Abonar $2.500.000 COP a Capital (Ley 1328 de 2009): Reducción de Plazo vs. Cuota",
    template="plotly_white",
    height=420,
)
fig_ley1328.update_traces(textposition="outside")
fig_ley1328.show()